# 02 - Time-Varying Parameter VAR (TVP-VAR)

This notebook covers the **TVP-VAR** model, where VAR coefficients are allowed to evolve over time. This is essential for capturing **structural changes** in macroeconomic relationships — e.g., shifts in monetary policy regimes, changes in the Phillips curve, or the Great Moderation.

## Topics covered

- Constant vs time-varying parameters: why it matters
- The TVP-VAR state-space formulation
- Estimation via Kalman filter and smoother
- Stochastic volatility: time-varying error variances
- Visualizing coefficient evolution over time
- Time-varying IRFs: how the transmission mechanism changes
- Exercises: detecting structural breaks via TVP-VAR

---

## The TVP-VAR Model

The standard VAR assumes constant coefficients. The TVP-VAR relaxes this:

**Observation equation:**
$$Y_t = X_t \beta_t + \varepsilon_t, \quad \varepsilon_t \sim N(0, H)$$

**State equation (random walk):**
$$\beta_t = \beta_{t-1} + u_t, \quad u_t \sim N(0, Q)$$

where:
- $\beta_t$ is the vectorized form of all VAR coefficients at time $t$
- $X_t$ is the regressor matrix (lags + intercept)
- $H$ is the observation noise covariance (constant or time-varying)
- $Q$ controls the **speed of parameter drift** — larger $Q$ means faster evolution

The Kalman filter provides optimal filtering (real-time estimates $\beta_{t|t}$), while the Rauch-Tung-Striebel smoother provides smoothed estimates $\beta_{t|T}$ using the full sample.

**Key references:**
- Primiceri, G.E. (2005). "Time Varying Structural Vector Autoregressions and Monetary Policy." *Review of Economic Studies*, 72(3), 821-852.
- Cogley, T. & Sargent, T.J. (2005). "Drifts and Volatilities: Monetary Policies and Outcomes in the Post WWII US." *Review of Economic Dynamics*, 8(2), 262-302.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox import TVPVAR

sys.path.insert(0, os.path.join("..", "utils"))
from data_generators import generate_tvp_var
from plot_helpers import plot_tvp_coefficients

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Synthetic TVP-VAR Data

We first generate data from a **known** TVP-VAR process where the coefficients evolve as random walks. This allows us to verify that the Kalman filter recovers the true coefficient paths.

In [ ]:
# Generate synthetic TVP-VAR data
Y_synth, A_path_true = generate_tvp_var(n=200, k=2, seed=42)

print(f"Generated data shape: {Y_synth.shape}  (T, K)")
print(f"True coefficient path shape: {A_path_true.shape}  (T, K, K)")
print(f"\nInitial A[0]:")
print(A_path_true[0].round(4))
print(f"\nFinal A[T-1]:")
print(A_path_true[-1].round(4))

# Visualize the data
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
var_names = ["Variable 1", "Variable 2"]
colors = ["steelblue", "firebrick"]

for i, (ax, name, color) in enumerate(zip(axes, var_names, colors)):
    ax.plot(Y_synth[:, i], color=color, linewidth=0.8)
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)

plt.suptitle("Synthetic TVP-VAR Data", fontsize=14)
plt.xlabel("Observation")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the true coefficient evolution
fig = plot_tvp_coefficients(
    A_path_true,
    variable_names=["Y1", "Y2"],
    title="True Time-Varying Coefficients A_t",
)
plt.savefig(os.path.join("..", "outputs", "tvpvar_true_coefficients.png"), bbox_inches="tight")
plt.show()

## 2. Kalman Filter Estimation

The `chronobox.TVPVAR` class implements the full Kalman filter and Rauch-Tung-Striebel smoother:

1. **Prediction step:** $\beta_{t|t-1} = \beta_{t-1|t-1}$, $P_{t|t-1} = P_{t-1|t-1} + Q$
2. **Update step:** uses the innovation $v_t = Y_t - X_t \beta_{t|t-1}$ and Kalman gain to update
3. **Smoothing:** backward pass refines estimates using future information

The key hyperparameter is `Q_scale`, which controls how quickly coefficients can drift. Larger values allow faster adaptation but noisier estimates.

In [ ]:
# Fit TVP-VAR on synthetic data
tvp_model = TVPVAR(lags=1, Q_scale=0.01)
tvp_results = tvp_model.fit(Y_synth)

print(f"TVP-VAR estimation complete")
print(f"  Number of observations: {tvp_results.n_obs}")
print(f"  Number of variables: {tvp_results.k_vars}")
print(f"  Lags: {tvp_results.lags}")
print(f"  Log-likelihood: {tvp_results.log_likelihood:.2f}")
print(f"  Time-varying coefs shape: {tvp_results.coefs_time.shape}")
print(f"  Filtered state shape: {tvp_results.beta_filtered.shape}")
print(f"  Smoothed state shape: {tvp_results.beta_smoothed.shape}")
print(f"\nObservation noise covariance (H = Sigma_OLS):")
print(tvp_results.sigma.round(4))

## 3. Visualizing Time-Varying Coefficients

The core output of TVP-VAR is the **coefficient path** $A_t$. For a bivariate VAR(1), we have four coefficients that evolve over time. Let us compare the estimated paths with the true ones.

In [ ]:
# Compare true vs estimated coefficient paths
# The true path has T=200 observations; TVP-VAR drops the first p=1 lag
T_tvp = tvp_results.n_obs
A_true_aligned = A_path_true[1:]  # align: skip first observation (used as initial lag)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
labels = [("Y1", "Y1"), ("Y1", "Y2"), ("Y2", "Y1"), ("Y2", "Y2")]

for idx, (i, j) in enumerate([(0,0), (0,1), (1,0), (1,1)]):
    ax = axes.flat[idx]
    
    # True path
    ax.plot(A_true_aligned[:T_tvp, i, j], color="steelblue", linewidth=1.5, 
            label="True", alpha=0.7)
    
    # Filtered estimate
    est_path = tvp_results.coefficient_path(lag=0, i=i, j=j)
    ax.plot(est_path, color="firebrick", linewidth=1.5, 
            label="Filtered", linestyle="--")
    
    ax.set_title(f"A[{labels[idx][0]}, {labels[idx][1]}]", fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

plt.suptitle("TVP-VAR: True vs Filtered Coefficient Paths", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "tvpvar_coefficient_recovery.png"), bbox_inches="tight")
plt.show()

## 4. Stochastic Volatility and Q_scale Sensitivity

In the full Primiceri (2005) model, both the **coefficients** and the **error covariance** evolve over time (stochastic volatility). While our implementation uses a constant $H$, the `Q_scale` parameter controls how much the coefficients drift.

- **Small Q_scale** (e.g., 0.001): coefficients are nearly constant → similar to standard VAR
- **Large Q_scale** (e.g., 0.1): coefficients change rapidly → may overfit noise

Let us compare different Q_scale values.

In [ ]:
# Compare Q_scale sensitivity
q_scales = [0.001, 0.01, 0.05, 0.1]
results_by_q = {}

for q in q_scales:
    model_q = TVPVAR(lags=1, Q_scale=q)
    res_q = model_q.fit(Y_synth)
    results_by_q[q] = res_q
    print(f"Q_scale={q:.3f}: log-likelihood = {res_q.log_likelihood:.2f}")

# Compare coefficient A[0,0] path for different Q_scale
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
colors_q = {0.001: "steelblue", 0.01: "firebrick", 0.05: "darkgreen", 0.1: "darkorange"}

for idx, (i, j) in enumerate([(0,0), (0,1), (1,0), (1,1)]):
    ax = axes.flat[idx]
    
    # True path
    ax.plot(A_true_aligned[:T_tvp, i, j], color="black", linewidth=2, 
            label="True", alpha=0.5)
    
    for q, res_q in results_by_q.items():
        path = res_q.coefficient_path(lag=0, i=i, j=j)
        ax.plot(path, color=colors_q[q], linewidth=1.2, label=f"Q={q}")
    
    ax.set_title(f"A[{i},{j}]", fontsize=11)
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(fontsize=8, loc="best")

plt.suptitle("TVP-VAR: Q_scale Sensitivity (Stochastic Volatility Proxy)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "tvpvar_qscale_sensitivity.png"), bbox_inches="tight")
plt.show()

## 5. Application to US Macro Data

Now we apply the TVP-VAR to real macroeconomic data. A key application is studying whether the **monetary transmission mechanism** has changed over time — for example, has the effect of interest rate changes on GDP and inflation changed between the Volcker era and the Great Moderation?

In [ ]:
# Load US macro data
data_path = os.path.join("..", "data", "us_macro_quarterly.csv")
df = pd.read_csv(data_path, parse_dates=["date"])
df.set_index("date", inplace=True)

# Use 3 variables for TVP-VAR: inflation, gdp, fed_funds
macro_vars = ["inflation", "gdp", "fed_funds"]
endog = df[macro_vars].values
dates = df.index

print(f"Data shape: {endog.shape}")
print(f"Period: {dates[0]} to {dates[-1]}")
print(f"Variables: {macro_vars}")
df[macro_vars].describe().round(3)

In [ ]:
# Fit TVP-VAR(1) on US macro data
tvp_macro = TVPVAR(lags=1, Q_scale=0.01)
res_macro = tvp_macro.fit(endog)

print(f"TVP-VAR estimation on US macro data:")
print(f"  Observations: {res_macro.n_obs}")
print(f"  Variables: {res_macro.k_vars}")
print(f"  Log-likelihood: {res_macro.log_likelihood:.2f}")

# Visualize time-varying coefficients
dates_tvp = dates[1:]  # drop first p=1 observations
fig = plot_tvp_coefficients(
    res_macro.coefs_time[:, 0, :, :],  # first (and only) lag
    variable_names=macro_vars,
    dates=dates_tvp[:res_macro.n_obs],
    title="TVP-VAR(1): Time-Varying Coefficients (US Macro)",
)
plt.savefig(os.path.join("..", "outputs", "tvpvar_us_macro_coefficients.png"), bbox_inches="tight")
plt.show()

## 6. Time-Varying IRFs

A powerful feature of TVP-VAR is computing IRFs **at different time points**. This reveals how the transmission mechanism has evolved. For instance, we can compare the monetary policy IRF during:

- **Early period** (e.g., t=20, late 1970s/early 1980s — Volcker disinflation)
- **Middle period** (e.g., t=100, mid-1990s — Great Moderation)
- **Late period** (e.g., near end of sample — recent era)

In [ ]:
# Compute time-varying IRFs at three different time points
T_total = res_macro.n_obs
time_points = {
    "Early (t=20)": 20,
    "Middle (t=100)": min(100, T_total - 1),
    "Late (t=T-5)": T_total - 5,
}

irf_by_time = {}
for label, t in time_points.items():
    irf_t = res_macro.time_varying_irf(t=t, periods=16, identification="cholesky")
    irf_by_time[label] = irf_t
    date_label = dates_tvp[t].strftime("%Y") if t < len(dates_tvp) else f"t={t}"
    print(f"{label} (approx. {date_label}): IRF shape = {irf_t.shape}")

# Plot: response of each variable to a fed_funds shock at different time points
fed_funds_idx = 2  # fed_funds is the 3rd variable
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_time = {"Early (t=20)": "steelblue", "Middle (t=100)": "firebrick", "Late (t=T-5)": "darkgreen"}

for i, (ax, var_name) in enumerate(zip(axes, macro_vars)):
    for label, irf_t in irf_by_time.items():
        horizons = np.arange(irf_t.shape[0])
        ax.plot(horizons, irf_t[:, i, fed_funds_idx], 
                color=colors_time[label], linewidth=1.8, label=label)
    
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_title(f"Response of {var_name}", fontsize=11)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

plt.suptitle("TVP-VAR: Time-Varying IRF to Monetary Policy Shock", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "tvpvar_time_varying_irf.png"), bbox_inches="tight")
plt.show()

## 7. Detecting Structural Changes

By examining the **time path of specific coefficients**, we can detect structural changes. For example, the coefficient on lagged inflation in the fed_funds equation captures how aggressively the central bank responds to inflation (the "Taylor rule" coefficient).

In [ ]:
# Extract key economic coefficients over time
# Equation for fed_funds (index 2):
#   fed_funds_t = a20 * inflation_{t-1} + a21 * gdp_{t-1} + a22 * fed_funds_{t-1} + intercept

# Taylor rule coefficient: response of fed_funds to inflation
taylor_inflation = res_macro.coefficient_path(lag=0, i=2, j=0)
# Response of fed_funds to gdp
taylor_gdp = res_macro.coefficient_path(lag=0, i=2, j=1)
# Persistence of fed_funds (interest rate smoothing)
rate_smoothing = res_macro.coefficient_path(lag=0, i=2, j=2)
# Intercept of fed_funds equation
intercept_ff = res_macro.intercept_time[:, 2]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
t_axis = dates_tvp[:res_macro.n_obs]

# Taylor rule: inflation response
axes[0, 0].plot(t_axis, taylor_inflation, color="firebrick", linewidth=1.5)
axes[0, 0].axhline(0, color="k", linewidth=0.5)
axes[0, 0].set_title("Fed Funds → Inflation (Taylor rule)", fontsize=10)
axes[0, 0].set_ylabel("Coefficient")
axes[0, 0].grid(True, alpha=0.3)

# Taylor rule: GDP response
axes[0, 1].plot(t_axis, taylor_gdp, color="steelblue", linewidth=1.5)
axes[0, 1].axhline(0, color="k", linewidth=0.5)
axes[0, 1].set_title("Fed Funds → GDP (output stabilization)", fontsize=10)
axes[0, 1].set_ylabel("Coefficient")
axes[0, 1].grid(True, alpha=0.3)

# Interest rate smoothing
axes[1, 0].plot(t_axis, rate_smoothing, color="darkgreen", linewidth=1.5)
axes[1, 0].axhline(0, color="k", linewidth=0.5)
axes[1, 0].set_title("Fed Funds → Lagged Fed Funds (rate smoothing)", fontsize=10)
axes[1, 0].set_ylabel("Coefficient")
axes[1, 0].grid(True, alpha=0.3)

# Phillips curve: inflation persistence
infl_persistence = res_macro.coefficient_path(lag=0, i=0, j=0)
axes[1, 1].plot(t_axis, infl_persistence, color="darkorange", linewidth=1.5)
axes[1, 1].axhline(0, color="k", linewidth=0.5)
axes[1, 1].set_title("Inflation Persistence (own lag)", fontsize=10)
axes[1, 1].set_ylabel("Coefficient")
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("TVP-VAR: Key Economic Coefficients Over Time", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "tvpvar_structural_change.png"), bbox_inches="tight")
plt.show()

## 8. IRF Surface: Continuous Time-Varying Response

We can visualize the IRF as a **3D surface** where one axis is the horizon, another is time, and the height is the response. This gives a complete picture of how the transmission mechanism evolves.

In [ ]:
# Compute IRF at every 10th time point for a surface plot
step = 10
time_indices = range(5, res_macro.n_obs - 5, step)
n_horizons = 12
irf_surface = np.zeros((len(list(time_indices)), n_horizons + 1, 3, 3))

time_list = list(time_indices)
for idx, t in enumerate(time_list):
    irf_surface[idx] = res_macro.time_varying_irf(t=t, periods=n_horizons, identification="cholesky")

# Heatmap: response of GDP to fed_funds shock over time x horizon
response_gdp_ff = irf_surface[:, :, 1, 2]  # GDP (idx 1) response to ff (idx 2)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(response_gdp_ff.T, aspect="auto", cmap="RdBu_r",
               vmin=-np.max(np.abs(response_gdp_ff)),
               vmax=np.max(np.abs(response_gdp_ff)),
               origin="lower")

# Label axes with approximate dates
tick_positions = range(0, len(time_list), max(1, len(time_list) // 8))
tick_labels = [dates_tvp[time_list[i]].strftime("%Y") if time_list[i] < len(dates_tvp) 
               else str(time_list[i]) for i in tick_positions]
ax.set_xticks(list(tick_positions))
ax.set_xticklabels(tick_labels)
ax.set_yticks(range(0, n_horizons + 1, 2))
ax.set_yticklabels(range(0, n_horizons + 1, 2))

ax.set_xlabel("Time (estimation date)")
ax.set_ylabel("Horizon (quarters)")
ax.set_title("TVP-VAR: Response of GDP to Fed Funds Shock (Time × Horizon)", fontsize=13)
fig.colorbar(im, ax=ax, label="Response", shrink=0.8)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "tvpvar_irf_surface.png"), bbox_inches="tight")
plt.show()

## Exercicio

### Exercicio 1: Detectar mudancas estruturais via TVP-VAR

A "Great Moderation" (aprox. 1984 em diante) e frequentemente associada a uma reducao na volatilidade macroeconomica e mudancas na conducao da politica monetaria.

1. Estime um TVP-VAR(2) com as 3 variaveis macro e `Q_scale=0.02`
2. Extraia o coeficiente de resposta do fed_funds a inflacao (coeficiente de Taylor) ao longo do tempo
3. O coeficiente aumentou apos 1984? Isso seria consistente com uma politica monetaria mais agressiva contra inflacao
4. Compare os IRFs em 1980 vs 1995 vs 2010 — como mudou a resposta do GDP a um choque monetario?

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

# Dica:
# tvp_ex = TVPVAR(lags=2, Q_scale=0.02)
# res_ex = tvp_ex.fit(endog)
# taylor_coef = res_ex.coefficient_path(lag=0, i=2, j=0)  # ff -> inflation
# 
# # Find the observation closest to 1984
# idx_1984 = np.argmin(np.abs(dates_tvp[:res_ex.n_obs] - pd.Timestamp("1984-01-01")))
# print(f"Taylor coefficient pre-1984 avg: {taylor_coef[:idx_1984].mean():.4f}")
# print(f"Taylor coefficient post-1984 avg: {taylor_coef[idx_1984:].mean():.4f}")

### Exercicio 2: Comparacao TVP-VAR vs VAR constante

Compare as previsoes do TVP-VAR com um VAR com parametros constantes:

1. Estime ambos os modelos nos mesmos dados
2. Compute o erro quadratico medio (MSE) das previsoes 1-passo-a-frente de ambos
3. O TVP-VAR melhora a previsao? Em quais periodos a melhoria e maior?
4. Plote a diferenca de MSE ao longo do tempo — ela e maior em periodos de mudanca estrutural?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

# Dica:
# from chronobox import VAR
# var_const = VAR(lags=1, trend="c")
# var_res = var_const.fit(endog)
# 
# # 1-step-ahead errors from constant VAR
# mse_const = np.mean(var_res.resid**2, axis=1)
# 
# # 1-step-ahead errors from TVP-VAR (use filtered coefficients)
# # For each t, the 1-step prediction is X_t @ beta_{t|t-1}
# # The TVP-VAR residual is Y_t - X_t @ beta_{t|t}

---

## Resumo

Neste notebook aprendemos:

1. **TVP-VAR**: permite que os coeficientes do VAR evoluam no tempo como passeio aleatorio, capturando mudancas estruturais na economia

2. **Filtro de Kalman**: algoritmo otimo para estimar os estados latentes (coeficientes) dadas as observacoes. O smoother (RTS) refina as estimativas usando toda a amostra

3. **Volatilidade estocastica**: o parametro `Q_scale` controla a velocidade de variacao dos coeficientes — analogia com volatilidade estocastica no modelo completo de Primiceri (2005)

4. **IRFs variantes no tempo**: permitem estudar como o mecanismo de transmissao da politica monetaria mudou ao longo das decadas

5. **Deteccao de mudancas estruturais**: a evolucao temporal dos coeficientes revela mudancas de regime (ex: Great Moderation, mudancas na regra de Taylor)

## Referencias

- Primiceri, G.E. (2005). "Time Varying Structural Vector Autoregressions and Monetary Policy." *Review of Economic Studies*, 72(3), 821-852.
- Cogley, T. & Sargent, T.J. (2005). "Drifts and Volatilities: Monetary Policies and Outcomes in the Post WWII US." *Review of Economic Dynamics*, 8(2), 262-302.
- Koop, G. & Korobilis, D. (2013). "Large Time-Varying Parameter VARs." *JBES*, 31(3), 265-279.

No proximo notebook, veremos o **GVAR** e **Historical Decomposition** — modelos multi-pais e decomposicao de choques.